# Signal — Weekly Momentum Ranking

Reads `semiconductor_data.csv` (from Data) and produces weekly long/short picks:
top 5 momentum winners (long), bottom 5 (short), for each trading week.

**Decisions locked in below (flag if the team wants to change these):**
- **Lookback window:** 20 trading days trailing return (matches `avg_volume_20d`'s
  window for consistency). Configurable via `LOOKBACK_DAYS`.
- **Timing convention:** the score for trading week `W` is computed from the
  **closing price on the last trading day of week `W-1`**. Positions are assumed
  to open at **week `W`'s first-day open** (Monday open, or the first trading day
  if Monday was a holiday). Execution should use `week_number`-matched `open`
  prices from `semiconductor_data.csv` as the entry price — **never** a price
  from inside the signal window itself, that would be lookahead bias.
- **Tie-break:** higher `avg_volume_20d` wins (prefer the more liquid name),
  then alphabetical by ticker, for full determinism.
- **Edge cases:** a ticker missing a valid momentum score that week (not enough
  trailing history yet, or a data gap) is dropped from that week's ranking pool.
  If fewer than 10 tickers remain valid, the whole week is skipped (no picks
  output) rather than trading a partial/unreliable universe.

Output: `signal_picks.csv`, one row per `(week_number, ticker)` for the 10
selected names each week — columns `week_number, signal_date, ticker, side,
rank, momentum_score`.

In [1]:
# ============================================================
# CONFIG
# ============================================================
import pandas as pd
import numpy as np

INPUT_CSV = "semiconductor_data.csv"
OUTPUT_CSV = "signal_picks.csv"

LOOKBACK_DAYS = 20      # trailing-return window for the momentum score (trading days)
N_LONG = 5              # top N by momentum -> long
N_SHORT = 5             # bottom N by momentum -> short
MIN_VALID_TICKERS = N_LONG + N_SHORT  # need at least this many valid scores to trade a week

print(f"Lookback: {LOOKBACK_DAYS} trading days | Long: {N_LONG} | Short: {N_SHORT}")

Lookback: 20 trading days | Long: 5 | Short: 5


In [2]:
# ============================================================
# 1. LOAD THE DATA CONTRACT
# ============================================================
df = pd.read_csv(INPUT_CSV, parse_dates=["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

expected_cols = {"date", "ticker", "open", "high", "low", "close", "volume",
                  "avg_volume_20d", "week_number"}
missing_cols = expected_cols - set(df.columns)
assert not missing_cols, f"semiconductor_data.csv is missing columns: {missing_cols}"

n_tickers = df["ticker"].nunique()
n_weeks = df["week_number"].nunique()
print(f"Loaded {len(df)} rows | {n_tickers} tickers | weeks {df['week_number'].min()}-{df['week_number'].max()}")

Loaded 10040 rows | 20 tickers | weeks 1-105


In [3]:
# ============================================================
# 2. MOMENTUM SCORE (trailing return, strictly backward-looking)
# ============================================================
# For each ticker, momentum_score on date D = close[D] / close[D - LOOKBACK_DAYS] - 1,
# using only that ticker's own price history up to and including D. Because this
# is a shift() over already-sorted-by-date rows, it can NEVER see a future price
# -- this is what keeps the whole pipeline lookahead-free.
df["momentum_score"] = (
    df.groupby("ticker")["close"]
    .transform(lambda s: s / s.shift(LOOKBACK_DAYS) - 1)
)

n_valid = df["momentum_score"].notna().sum()
print(f"Momentum score computed for {n_valid}/{len(df)} rows "
      f"(first {LOOKBACK_DAYS} trading days per ticker are NaN -- not enough history yet).")

Momentum score computed for 9640/10040 rows (first 20 trading days per ticker are NaN -- not enough history yet).


In [4]:
# ============================================================
# 3. BUILD THE WEEKLY SIGNAL (score at end of week W-1 -> trade week W)
# ============================================================
# The score used to trade week W must come only from data through the end of
# week W-1 -- using anything from inside week W itself would be lookahead bias.
# "asof_date" = the last trading day of each week (its Friday close, or the
# last day that week if Friday was a holiday).
week_asof = df.groupby("week_number")["date"].max()
asof_dates = set(week_asof)

signal_source = df[df["date"].isin(asof_dates)][
    ["week_number", "date", "ticker", "momentum_score", "avg_volume_20d"]
].rename(columns={"date": "signal_date"})

# Shift the source week forward by one: a score computed at the end of week W
# becomes the signal for trading week W+1.
trade_signals = signal_source.copy()
trade_signals["week_number"] = trade_signals["week_number"] + 1

# Drop any "trade week" beyond the last week we actually have data for --
# there's no real week to trade it in, so it's not a usable signal.
max_week = df["week_number"].max()
trade_signals = trade_signals[trade_signals["week_number"] <= max_week]

print(f"Built signals for trade weeks {trade_signals['week_number'].min()}-{trade_signals['week_number'].max()}")

Built signals for trade weeks 2-105


In [5]:
# ============================================================
# 4. RANK AND SELECT TOP 5 / BOTTOM 5 PER WEEK
# ============================================================
# Tie-break rule (applied consistently to both sides): higher avg_volume_20d
# wins (prefer the more liquid name), then alphabetical by ticker.
TIE_BREAK_COLS = ["avg_volume_20d", "ticker"]
TIE_BREAK_ASC = [False, True]

picks = []
skipped_weeks = []  # (week_number, n_valid_tickers) for weeks we couldn't trade

for wk, grp in trade_signals.groupby("week_number"):
    valid = grp.dropna(subset=["momentum_score"])  # drop tickers missing a score this week

    if len(valid) < MIN_VALID_TICKERS:
        skipped_weeks.append((wk, len(valid)))
        continue

    ranked = valid.sort_values(
        ["momentum_score"] + TIE_BREAK_COLS,
        ascending=[False] + TIE_BREAK_ASC,
    ).reset_index(drop=True)

    longs = ranked.head(N_LONG).copy()
    longs["side"] = "long"
    longs["rank"] = range(1, len(longs) + 1)

    shorts = ranked.tail(N_SHORT).sort_values(
        ["momentum_score"] + TIE_BREAK_COLS,
        ascending=[True] + TIE_BREAK_ASC,
    ).reset_index(drop=True).copy()
    shorts["side"] = "short"
    shorts["rank"] = range(1, len(shorts) + 1)

    picks.append(longs)
    picks.append(shorts)

signal_df = pd.concat(picks, ignore_index=True)

print(f"Picks generated for {signal_df['week_number'].nunique()} weeks.")
if skipped_weeks:
    print(f"Skipped {len(skipped_weeks)} weeks (fewer than {MIN_VALID_TICKERS} valid tickers): {skipped_weeks}")
else:
    print("No weeks skipped.")

Picks generated for 100 weeks.
Skipped 4 weeks (fewer than 10 valid tickers): [(2, 0), (3, 0), (4, 0), (5, 0)]


In [6]:
# ============================================================
# 5. FINALIZE COLUMNS + SAVE
# ============================================================
final_cols = ["week_number", "signal_date", "ticker", "side", "rank", "momentum_score"]
signal_df = signal_df[final_cols].sort_values(["week_number", "side", "rank"]).reset_index(drop=True)
signal_df["signal_date"] = signal_df["signal_date"].dt.strftime("%Y-%m-%d")

signal_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(signal_df)} rows to {OUTPUT_CSV}")

Saved 1000 rows to signal_picks.csv


In [7]:
# ============================================================
# 6. SANITY CHECKS -- read this before handing off to Execution
# ============================================================
print("=" * 60)
print("SAMPLE OUTPUT (first week with picks)")
print("=" * 60)
first_week = signal_df["week_number"].min()
print(signal_df[signal_df["week_number"] == first_week])

print("\n" + "=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# 1. No NaNs in the output
nan_counts = signal_df.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
print(f"[{'OK' if len(nan_counts) == 0 else 'FAIL'}] NaNs in output: "
      f"{'none' if len(nan_counts) == 0 else dict(nan_counts)}")

# 2. Every non-skipped week has exactly N_LONG + N_SHORT picks
counts = signal_df.groupby("week_number").size()
bad_counts = counts[counts != (N_LONG + N_SHORT)]
print(f"[{'OK' if len(bad_counts) == 0 else 'FAIL'}] Weeks with wrong pick count: "
      f"{'none' if len(bad_counts) == 0 else dict(bad_counts)}")

# 3. Do longs actually outscore shorts on momentum? (directional sanity check)
avg_by_side = signal_df.groupby("side")["momentum_score"].mean()
longs_beat_shorts = avg_by_side.get("long", -np.inf) > avg_by_side.get("short", np.inf)
print(f"[{'OK' if longs_beat_shorts else 'FAIL'}] Avg momentum -- long: "
      f"{avg_by_side.get('long', float('nan')):.2%}, short: {avg_by_side.get('short', float('nan')):.2%}")

# 4. Week-to-week turnover -- picks should change, not stay static.
#    Overlap of 5 means "fully static" (bad); overlap of 0 means "fully different".
turnover_records = []
weeks_sorted = sorted(signal_df["week_number"].unique())
for prev_wk, cur_wk in zip(weeks_sorted, weeks_sorted[1:]):
    if cur_wk != prev_wk + 1:
        continue  # not consecutive (a week was skipped) -- not a fair comparison
    for side in ["long", "short"]:
        prev_set = set(signal_df[(signal_df.week_number == prev_wk) & (signal_df.side == side)]["ticker"])
        cur_set = set(signal_df[(signal_df.week_number == cur_wk) & (signal_df.side == side)]["ticker"])
        turnover_records.append({"side": side, "overlap_of_5": len(prev_set & cur_set)})

turnover_df = pd.DataFrame(turnover_records)
if len(turnover_df) > 0:
    avg_overlap = turnover_df.groupby("side")["overlap_of_5"].mean()
    fully_static = (turnover_df["overlap_of_5"] == 5).mean()
    print(f"[INFO] Avg week-over-week overlap (out of 5 names): "
          f"long={avg_overlap.get('long', float('nan')):.1f}, short={avg_overlap.get('short', float('nan')):.1f}")
    print(f"[{'OK' if fully_static < 0.5 else 'FAIL'}] Fraction of weeks fully static (all 5 unchanged): {fully_static:.0%}")
else:
    print("[INFO] Not enough consecutive weeks to measure turnover.")

# 5. Skipped weeks summary
print(f"[INFO] Weeks skipped (insufficient history/data): {len(skipped_weeks)} "
      f"out of {trade_signals['week_number'].nunique()} candidate weeks.")

SAMPLE OUTPUT (first week with picks)
   week_number signal_date ticker   side  rank  momentum_score
0            6  2024-10-04   AVGO   long     1        0.293584
1            6  2024-10-04    AMD   long     2        0.272051
2            6  2024-10-04   NVDA   long     3        0.214924
3            6  2024-10-04   INTC   long     4        0.195871
4            6  2024-10-04     MU   long     5        0.183723
5            6  2024-10-04   SWKS  short     1       -0.019840
6            6  2024-10-04    STM  short     2        0.005357
7            6  2024-10-04    TXN  short     3        0.026640
8            6  2024-10-04   NXPI  short     4        0.037818
9            6  2024-10-04   MCHP  short     5        0.048455

SANITY CHECKS
[OK] NaNs in output: none
[OK] Weeks with wrong pick count: none
[OK] Avg momentum -- long: 17.74%, short: -8.12%
[INFO] Avg week-over-week overlap (out of 5 names): long=3.4, short=3.3
[OK] Fraction of weeks fully static (all 5 unchanged): 7%
[INFO] Wee

In [8]:
# ============================================================
# 7. DOES THE SIGNAL ACTUALLY PREDICT FORWARD RETURNS? (the real test)
# ============================================================
# Section 6's "long beats short on momentum_score" check is circular -- that's
# literally how we picked them, it can't fail. The real question is whether
# LAST week's momentum predicts THIS week's realized return. We proxy realized
# weekly return as (close on the week's last trading day) / (open on the
# week's first trading day) - 1, matching our stated entry convention (enter
# at the week's open). This is a rough proxy for Signal's own use -- Execution
# and P&L own the real fill/P&L simulation.
week_first_date = df.groupby("week_number")["date"].min()
week_last_date = df.groupby("week_number")["date"].max()

opens_on_first_day = df[df["date"].isin(set(week_first_date))][["week_number", "ticker", "open"]] \
    .rename(columns={"open": "week_open_px"})
closes_on_last_day = df[df["date"].isin(set(week_last_date))][["week_number", "ticker", "close"]] \
    .rename(columns={"close": "week_close_px"})

weekly_fwd = opens_on_first_day.merge(closes_on_last_day, on=["week_number", "ticker"])
weekly_fwd["fwd_return"] = weekly_fwd["week_close_px"] / weekly_fwd["week_open_px"] - 1

eval_df = signal_df.merge(weekly_fwd[["week_number", "ticker", "fwd_return"]],
                           on=["week_number", "ticker"], how="left")

avg_fwd_by_side = eval_df.groupby("side")["fwd_return"].mean()
print(f"Realized avg weekly return -- long: {avg_fwd_by_side.get('long', float('nan')):.2%}, "
      f"short: {avg_fwd_by_side.get('short', float('nan')):.2%}")

weekly_spread = (
    eval_df[eval_df.side == "long"].groupby("week_number")["fwd_return"].mean()
    - eval_df[eval_df.side == "short"].groupby("week_number")["fwd_return"].mean()
)
hit_rate = (weekly_spread > 0).mean()
print(f"Weekly long-short spread: mean={weekly_spread.mean():.2%}, std={weekly_spread.std():.2%}, "
      f"hit rate (spread > 0) = {hit_rate:.0%} over {len(weekly_spread)} weeks")

Realized avg weekly return -- long: 1.00%, short: 0.47%
Weekly long-short spread: mean=0.53%, std=3.53%, hit rate (spread > 0) = 53% over 100 weeks


In [9]:
# ============================================================
# 8. LOOKBACK ROBUSTNESS CHECK
# ============================================================
# Does the long-short edge only show up at exactly LOOKBACK_DAYS=20, or does
# it hold up across nearby lookback choices too? If it only "works" for one
# specific number, that's a red flag for overfitting rather than a real effect.
# This does NOT change signal_picks.csv -- it's a diagnostic only.

def run_signal(base_df, lookback_days):
    """Re-run steps 2-4 for a given lookback, returning just (week_number, ticker, side)."""
    d = base_df.copy()
    d["mom"] = d.groupby("ticker")["close"].transform(lambda s: s / s.shift(lookback_days) - 1)

    asof = d.groupby("week_number")["date"].max()
    src = d[d["date"].isin(set(asof))][["week_number", "ticker", "mom", "avg_volume_20d"]]
    ts = src.copy()
    ts["week_number"] = ts["week_number"] + 1
    ts = ts[ts["week_number"] <= d["week_number"].max()]

    rows = []
    for wk, grp in ts.groupby("week_number"):
        valid = grp.dropna(subset=["mom"])
        if len(valid) < MIN_VALID_TICKERS:
            continue
        ranked = valid.sort_values(["mom", "avg_volume_20d", "ticker"], ascending=[False, False, True])
        rows.append(ranked.head(N_LONG).assign(side="long"))
        rows.append(ranked.tail(N_SHORT).assign(side="short"))

    if not rows:
        return pd.DataFrame(columns=["week_number", "ticker", "side"])
    return pd.concat(rows, ignore_index=True)[["week_number", "ticker", "side"]]


robustness_rows = []
for lb in [10, 20, 40, 60]:
    picks_lb = run_signal(df, lb)
    ev = picks_lb.merge(weekly_fwd[["week_number", "ticker", "fwd_return"]],
                         on=["week_number", "ticker"], how="left")
    spread_lb = (
        ev[ev.side == "long"].groupby("week_number")["fwd_return"].mean()
        - ev[ev.side == "short"].groupby("week_number")["fwd_return"].mean()
    )
    robustness_rows.append({
        "lookback_days": lb,
        "weeks_traded": len(spread_lb),
        "avg_weekly_spread": spread_lb.mean(),
        "hit_rate": (spread_lb > 0).mean(),
    })

robustness_df = pd.DataFrame(robustness_rows)
print(robustness_df.to_string(index=False, formatters={
    "avg_weekly_spread": "{:.2%}".format,
    "hit_rate": "{:.0%}".format,
}))

n_positive = (robustness_df["avg_weekly_spread"] > 0).sum()
print(f"\n{n_positive}/{len(robustness_df)} lookback windows show a positive average weekly spread.")
if n_positive == len(robustness_df):
    print("Directionally consistent across lookbacks -- the momentum edge looks reasonably robust, "
          "not a fluke of picking exactly 20 days.")
elif n_positive <= len(robustness_df) // 2:
    print("Inconsistent across lookbacks -- possible overfitting to LOOKBACK_DAYS=20. Worth a team discussion "
          "before trusting this signal, or narrowing the universe/holding period.")
else:
    print("Mixed results -- the edge may be real but weak, or sensitive to the exact lookback chosen.")

 lookback_days  weeks_traded avg_weekly_spread hit_rate
            10           102             0.10%      50%
            20           100             0.53%      53%
            40            96             0.35%      55%
            60            92            -0.09%      54%

3/4 lookback windows show a positive average weekly spread.
Mixed results -- the edge may be real but weak, or sensitive to the exact lookback chosen.
